In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

plt.rcParams["figure.figsize"] = (12, 6)

# 1) Dataset
DATA_PATH = Path(r"C:\Users\river\OneDrive\Escritorio\Pablo\Iron Hack\Labs\lab-eda-univariate\amz_uk_price_prediction_dataset.csv")
df = pd.read_csv(DATA_PATH)

# 2) Column Category
category_col = "category"
if category_col is None:
    raise ValueError("No column with name category")

# 3) Category column cleaning
df[category_col] = (
    df[category_col].astype(str).str.strip().replace({"": np.nan}).fillna("Unknown")
)

print("Rows and columns:", df.shape)

In [ ]:
# FREQUENCY TABLES

# Absolute frecuency
freq_abs = df[category_col].value_counts(dropna=False).rename("absolute_frequency")
freq_rel = (df[category_col].value_counts(normalize=True, dropna=False) * 100)\
            .round(2).rename("relative_frequency_pct")

# Cumulative frecuency
freq_cum_abs = freq_abs.cumsum().rename("cumulative_abs")
freq_cum_rel = freq_rel.cumsum().rename("cumulative_rel_pct")

# Table
freq_table = (
    pd.concat([freq_abs, freq_rel, freq_cum_abs, freq_cum_rel], axis=1)
    .reset_index()
    .rename(columns={"index": category_col})
)

print("\n[Frequency Table]")
print(freq_table.head(5))

# Top-5
TOP_5 = 5
top5 = freq_table.head(TOP_5).copy()

print("\n[Top 5 Categories]")
for _, r in top5.iterrows():
    print(f"- {r[category_col]}: {int(r['absolute_frequency'])} ({r['relative_frequency_pct']}%)")

In [ ]:
# VISUALIZATIONS

# 1) Bar chart 
TOP_N = 10
topN = freq_table.head(TOP_N)

plt.figure()
plt.bar(topN[category_col].astype(str), topN["absolute_frequency"])
plt.xticks(rotation=45, ha="right")
plt.title(f"Top {TOP_N} Categories by listings")
plt.xlabel(category_col)
plt.ylabel("Listings")
plt.tight_layout()
plt.show()

# 2) Pie chart
plt.figure(figsize=(5,5))
plt.pie(
    top5["absolute_frequency"],
    labels=top5[category_col].astype(str),
    autopct="%1.1f%%",
    startangle=90,
    labeldistance=1.3,   # aleja las etiquetas del centro
    pctdistance=0.8      # ajusta distancia de los porcentajes
)
plt.title(f"Listings proportion — Top {TOP_5} categories")
plt.axis("equal")
plt.tight_layout()
plt.show()

In [6]:
price_col = "price"
df[price_col] = pd.to_numeric(df[price_col], errors="coerce")
price = df[price_col].dropna()
price = price[(price > 0) & (price < 1_000_000_000)]
print(f"[Prep] Column: {price_col} | Registros válidos: {price.size:,}")

[Prep] Column: price | Registros válidos: 2,443,631


In [ ]:
# CENTRALITY PRICING: mean, median, mode

mean_price = price.mean()
median_price = price.median()
mode_vals = price.mode()
mode_price = mode_vals.iloc[0] if not mode_vals.empty else np.nan

print("[Centrality]")
print(f"- Mean : {mean_price:,.2f}")
print(f"- Median : {median_price:,.2f}")
print(f"- Mode : {mode_price if pd.notna(mode_price) else 'N/A'}")

# Mean vs mode
if pd.notna(mode_price):
    diff = mean_price - mode_price
    print(f"- Diference mean - mode: {diff:,.2f}")

In [ ]:
# DISPERSION: var, std, range, IQR, min/max/Qs

# Variance y Standard Deviation
variance = price.var(ddof=1)
std_dev  = price.std(ddof=1)

# Min - Max - Range
pmin, pmax = price.min(), price.max()
prange = pmax - pmin

# Quantile - IQR
q1, q3 = price.quantile(0.25), price.quantile(0.75)
iqr = q3 - q1

print("[Dispersion]")
print(f"- Variance : {variance:,.2f}")
print(f"- Std Dev : {std_dev:,.2f}")
print(f"- Min: {pmin:,.2f} | Max: {pmax:,.2f} | Range: {prange:,.2f}")
print(f"- Q1: {q1:,.2f} | Q3: {q3:,.2f} | IQR: {iqr:,.2f}")

In [ ]:
plt.rcParams["figure.figsize"] = (10, 5)

# Helper Freedman–Diaconis (cap for performance)
def fd_bins_cap(values, cap=80):
    values = np.asarray(values)
    values = values[np.isfinite(values)]
    if values.size < 2:
        return 10
    iqr = np.subtract(*np.percentile(values, [75, 25]))
    if iqr == 0:
        return min(cap, 10)
    h = 2 * iqr / np.cbrt(values.size)
    if h <= 0:
        return min(cap, 10)
    bins = int(np.ceil((values.max() - values.min()) / h))
    return max(10, min(bins, cap))

def fast_hist_plot(values, bins):
    hist, bin_edges = np.histogram(values, bins=bins)
    widths = np.diff(bin_edges)
    lefts = bin_edges[:-1]
    plt.bar(lefts, hist, align="edge", width=widths)

# Histogram (raw) 
bins_raw = fd_bins_cap(price.values, cap=80)
plt.figure(figsize=(10,5))
fast_hist_plot(price.values, bins_raw)
plt.title(f"Histogram of {price_col} (raw)")
plt.xlabel("Price")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

# Box plot
sample_for_box = price.sample(n=min(150_000, price.size), random_state=42)
plt.figure(figsize=(8,5))
plt.boxplot(sample_for_box.values, vert=True, whis=1.5, showfliers=True)
plt.title(f"Box Plot of {price_col} (sampled)")
plt.ylabel("Price")
plt.tight_layout()
plt.show()

# 3) Histogram log10 — improves legibility
price_log = np.log10(price.values)
price_log = price_log[np.isfinite(price_log)]
bins_log = fd_bins_cap(price_log, cap=80)
plt.figure(figsize=(10,5))
fast_hist_plot(price_log, bins_log)
plt.title(f"Histogram of log10({price_col})")
plt.xlabel("log10(Price)")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()


In [ ]:
count = df["stars"].value_counts(dropna=False)
percent = df["stars"].value_counts(normalize=True, dropna=False) * 100

stars_table = pd.DataFrame({
    "count": count,
    "percentage": percent.round(2)
}).reset_index().rename(columns={"index": "stars_value"})

print(stars_table)

In [ ]:
rating_col = "stars"  

df[rating_col] = pd.to_numeric(df[rating_col], errors="coerce")
ratings = df[rating_col].dropna()
ratings = ratings[(ratings > 0) & (ratings <= 5)]

print(f"[Prep Ratings] Columnn: {rating_col} | Registers: {ratings.size:,}")

In [ ]:
# CENTRALITY RATINGS: mean, median, mode

mean_rt = ratings.mean()
median_rt = ratings.median()
mode_vals_rt = ratings.mode()
mode_rt = mode_vals_rt.iloc[0] if not mode_vals_rt.empty else np.nan

print(f"- Mean   : {mean_rt:,.2f}")
print(f"- Median : {median_rt:,.2f}")
print(f"- Mode   : {mode_rt if pd.notna(mode_rt) else 'N/A'}")
if pd.notna(mode_rt):
    diff_rt = mean_rt - mode_rt
    print(f"- Difference mean - mode: {diff_rt:,.2f}")

In [ ]:
# DISPERSION + SHAPE (var, std, IQR, skewness, kurtosis)

variance_rt = ratings.var(ddof=1)
std_rt      = ratings.std(ddof=1)

q1_rt, q3_rt = ratings.quantile(0.25), ratings.quantile(0.75)
iqr_rt = q3_rt - q1_rt

skew_rt = ratings.skew()
kurt_rt = ratings.kurt()

print("[Ratings — Dispersion & Shape]")
print(f"- Variance : {variance_rt:,.4f}")
print(f"- Std Dev  : {std_rt:,.4f}")
print(f"- Q1: {q1_rt:,.2f} | Q3: {q3_rt:,.2f} | IQR: {iqr_rt:,.2f}")
print(f"- Skewness : {skew_rt:,.4f}")
print(f"- Kurtosis : {kurt_rt:,.4f}")

In [ ]:
# Histogram ratings
bins = np.arange(0.5, 5.6, 0.5)

plt.figure(figsize=(8,5))
plt.hist(ratings.values, bins=bins, color="#3A86FF", edgecolor="black", rwidth=0.9)
plt.title("Histogram of Product Ratings (1–5 stars)")
plt.xlabel("Rating")
plt.ylabel("Frequency")
plt.xticks(np.arange(1, 5.5, 0.5))
plt.tight_layout()
plt.show()

The Exploratory Data Analysis of Amazon UK product listings reveals clear patterns in both pricing and customer ratings.

Product prices show a wide dispersion, with a large standard deviation and a huge range between the lowest and highest values. Most items are affordable, typically priced between £10 and £46, while a small fraction of very expensive products significantly increase the average price. This pattern produces a right-skewed distribution, confirmed visually through histograms and box plots that highlight numerous high outliers.

When applying a logarithmic transformation (log₁₀ of price), the distribution becomes more balanced, clearly showing the main concentration of prices in the low-to-mid range (£5–£40).

Regarding ratings, most products are rated between 4 and 5 stars, with 4.5 being the most common value, indicating a strong tendency toward high customer satisfaction. This concentration of high ratings and the small variation around the mean suggest that customers generally perceive the products very positively.

Overall, the analysis shows that while pricing is widely spread and skewed by a few high-value items, customer ratings are consistently high, reflecting a marketplace dominated by well-reviewed and competitively priced products.